# DeepWeeds Lab Day 2
Notebook tai lap: fold 0, chon cau hinh tren validation, test chi o cell chung ket. GPU va Internet can duoc bat tren Kaggle.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
repo = Path('/kaggle/working/K4-Track4-Day2-Deeplearning-Advance')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/kentranpr4-crypto/K4-Track4-Day2-Deeplearning-Advance.git', str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'scipy', 'thop', 'fvcore', 'openpyxl', 'scikit-learn'], check=True)
data = repo / 'data'; data.mkdir(exist_ok=True)
images = Path('/kaggle/input/datasets/anhtrangram/deepweeds-lab-assets/images')
if not images.is_dir(): raise FileNotFoundError('Attach deepweeds-lab-assets to Kaggle Input')
if not (data / 'images').exists(): (data / 'images').symlink_to(images, target_is_directory=True)
upstream = Path('/kaggle/working/DeepWeeds_labels')
if not upstream.exists(): subprocess.run(['git', 'clone', '-q', 'https://github.com/AlexOlsen/DeepWeeds.git', str(upstream)], check=True)
(data / 'labels').mkdir(exist_ok=True)
for file in (upstream / 'labels').glob('*.csv'): shutil.copy2(file, data / 'labels' / file.name)
env = os.environ.copy(); env['PYTHONPATH'] = str(repo); env['PYTHONUNBUFFERED'] = '1'
def run(*args):
    print('RUN', *args, flush=True)
    subprocess.run([sys.executable, '-u', *map(str, args)], cwd=repo, env=env, check=True)
print(repo)


## Du lieu va backbone
Moi backbone dung cung cong thuc, seed 0 va fold 0.


In [ ]:
run('starter/eda.py', '--images-dir', 'data/images', '--labels-dir', 'data/labels', '--out-dir', 'eda')
backbones = [('B01', 'resnet50'), ('B02', 'resnext50_32x4d'), ('B03', 'convnext_tiny'), ('B04', 'vit_small_patch16_224'), ('B05', 'efficientnet_b0')]
for exp_id, backbone in backbones:
    if not (repo / f'predictions/{exp_id}_seed0_val.csv').exists():
        last = repo / f'runs/{exp_id}/seed0/last.pt'
        run('starter/train.py', '--set', f'exp_id={exp_id}', f'backbone={backbone}', 'seed=0', 'epochs=12', 'batch_size=16', 'num_workers=0', 'amp=true', 'images_dir=data/images', 'labels_dir=data/labels', f'resume={str(last.exists()).lower()}')
run('starter/profile_backbones.py', '--experiments', *[name for name, _ in backbones])


## Ablation va chon suy luan
B03 duoc chon tu ket qua validation backbone. T01-T07 chi dung validation.


In [ ]:
run('starter/run_ablation.py', '--images-dir', 'data/images', '--labels-dir', 'data/labels', '--baseline-id', 'B03', '--backbone', 'convnext_tiny', '--seed', '0', '--epochs', '12', '--batch-size', '16', '--num-workers', '0')
run('starter/select_final.py', '--seed', '0', '--baseline-id', 'B03', '--measure-latency')
print((repo / 'selection_val.json').read_text())


## Chung ket
Chi chay sau khi selection_val.json da duoc chot. Test khong duoc dung de dieu chinh cau hinh.


In [ ]:
if not (repo / 'selection_val.json').exists(): raise FileNotFoundError('selection_val.json')
run('starter/run_final.py', '--images-dir', 'data/images', '--labels-dir', 'data/labels', '--seeds', '0', '1', '2')
run('starter/build_results.py', '--root', '.', '--xlsx', 'results.xlsx', '--report', 'report_draft.md')
for group in ('F01', 'T00'):
    for seed in (0, 1, 2):
        assert (repo / f'predictions/{group}_seed{seed}_test.csv').exists()
